Converting files to Markdown

In [ ]:
from docling.document_converter import DocumentConverter
import os

def load_pdfs_using_docling(file_paths, output_dir):
    converter=DocumentConverter()
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    # Handle directory path
    if isinstance(file_paths, str):
        if os.path.isdir(file_paths):
            # Get all PDF files from directory
            import glob
            file_paths = glob.glob(os.path.join(file_paths, "*.pdf"))
            print(f"Found {len(file_paths)} PDF files in directory")
        else:
            file_paths = [file_paths]
    for file_path in file_paths:
        if file_path.endswith('.pdf'):
            print(f"Converting {file_path} to txt..")
            try:
              result= converter.convert(file_path)
              document= result.document
              markdown_output=document.export_to_markdown()

              # create an output filename
              filename= os.path.basename(file_path)
              base_name= os.path.splitext(filename)[0]  # remove .pdf extention
              output_file= os.path.join(output_dir, f"{base_name}.md")
              with open(output_file, 'w') as f:
                  f.write(markdown_output)
              print(f"Successfully converted {filename} -> {base_name}.md")
            except Exception as e:
              print(f"Error converting {filepath}: {str(e)}")




In [2]:
! pip install huggingface-hub[cli] 


   ---------------------------------------- 2/2 [InquirerPy]



In [11]:
from huggingface_hub import HfApi
api= HfApi(token="hf_xxxx")
import json

with open(r"dataset_fixed.json", "r") as f:
    data= json.load(f)

api.upload_file(
    path_or_fileobj="dataset_fixed.json",
    path_in_repo="dataset_fixed.json",
    repo_id="sgaeyl/ChandlerHarryPotter",
    repo_type="dataset",
)

CommitInfo(commit_url='https://huggingface.co/datasets/sgaeyl/ChandlerHarryPotter/commit/d523bb42e410fa50e2fb22523fc340ef3477d935', commit_message='Upload dataset_fixed.json with huggingface_hub', commit_description='', oid='d523bb42e410fa50e2fb22523fc340ef3477d935', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sgaeyl/ChandlerHarryPotter', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sgaeyl/ChandlerHarryPotter'), pr_revision=None, pr_num=None)

In [9]:
import json
import os

def convert_to_json_array(input_file, output_file):
    """Convert multiple JSON objects to a single JSON array"""
    conversations = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        
        # Split by lines and process each JSON object
        lines = content.split('\n')
        current_json = ""
        
        for line in lines:
            line = line.strip()
            if line:
                current_json += line
                # Try to parse as complete JSON object
                try:
                    obj = json.loads(current_json)
                    conversations.append(obj)
                    current_json = ""
                except json.JSONDecodeError:
                    # Continue building the JSON string
                    continue
    
    # Write as proper JSON array
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(conversations, f, indent=2, ensure_ascii=False)
    
    print(f"Converted {len(conversations)} conversations to {output_file}")

# Alternative method if objects are clearly separated
def convert_separated_objects(input_file, output_file):
    """Convert file with separate JSON objects to array"""
    conversations = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        
        # Split by closing brace + opening brace pattern
        # This assumes objects are separated like: }{"conversations":
        objects = content.split('}\n{')
        
        for i, obj in enumerate(objects):
            # Add missing braces
            if i == 0:
                obj = obj + '}'
            elif i == len(objects) - 1:
                obj = '{' + obj
            else:
                obj = '{' + obj + '}'
            
            try:
                parsed = json.loads(obj)
                conversations.append(parsed)
            except json.JSONDecodeError as e:
                print(f"Error parsing object {i}: {e}")
                continue
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(conversations, f, indent=2, ensure_ascii=False)
    
    print(f"Converted {len(conversations)} conversations to {output_file}")

# Usage
if __name__ == "__main__":
    input_file = "NEWDATASET_question_answer_pairs.json"  # Your original file
    output_file = "dataset_fixed.json"  # Output file
    
    # Try method 1 first
    try:
        convert_to_json_array(input_file, output_file)
    except Exception as e:
        print(f"Method 1 failed: {e}")
        print("Trying method 2...")
        try:
            convert_separated_objects(input_file, output_file)
        except Exception as e:
            print(f"Method 2 failed: {e}")
            print("Manual inspection needed")

Converted 480 conversations to dataset_fixed.json


In [ ]:
import json
import os

# Ruta del notebook
ruta = "/content/bing_rag.ipynb"  # cambia esto

# Cargar el notebook como JSON
with open(ruta, "r", encoding="utf-8") as f:
    data = json.load(f)

# Limpia los widgets si están dañados
if "widgets" in data.get("metadata", {}):
    print("Corrigiendo metadata.widgets...")
    del data["metadata"]["widgets"]

# Guardar el notebook limpio
with open(ruta, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=1)

print("Notebook corregido y guardado.")